
# 🗽 NYC YELLOW TAXI DATA PIPELINE 🚖

---

Welcome to the NYC Taxi ETL pipeline notebook.<br>
This pipeline performs a robust transformation and loading process of the **Yellow Taxi** dataset as part of the NYCTAXI project.

---

**Pipeline Overview:**
1. **Source Table:** `NYCTAXI.BRONZE.YELLOW_TAXI`
2. **Key Steps:**
   - Read source data
   - Explore and validate timestamps
   - Filter for first half of 2026
   - Decode and transform key attributes
   - Save cleansed records to **Silver Table**
3. **Target Table:** `NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_CLEANSED`

---

> *Follow the steps below for clean, organized Yellow Taxi trip data ready for analytics.*

#### READ FROM INGESTION TABLE
- NYCTAXI.BRONZE.YELLOW_TAXI

In [0]:
from datetime import datetime

# =====================================================
# CALCULATE START TIME
# =====================================================

load_start_time = datetime.now()

In [0]:
nyc_yellow_taxi_df = spark.read.table('NYCTAXI.BRONZE.YELLOW_TAXI')
# nyc_yellow_taxi_df.display(3)
print(f'Records Effected: {nyc_yellow_taxi_df.count()}')

In [0]:
from pyspark.sql.functions import *

val_df = nyc_yellow_taxi_df.agg(max('tpep_pickup_datetime').alias('max_pickup_datetime'), 
                                min('tpep_pickup_datetime').alias('min_pickup_datetime')).display()

In [0]:
from pyspark.sql.functions import *

nyc_yellow_taxi_filtered_trans_df = nyc_yellow_taxi_df.filter((col("tpep_pickup_datetime") >= "2025-01-01") & (col("tpep_pickup_datetime") < "2026-07-01"))

In [0]:
from pyspark.sql.functions import col, when, unix_timestamp

nyc_yellow_taxi_trans_df = nyc_yellow_taxi_filtered_trans_df.select(
    when(col("VendorID") == 1, "Creative Mobile Technologies, LLC")
      .when(col("VendorID") == 2, "Curb Mobility, LLC")
      .when(col("VendorID") == 6, "Myle Technologies Inc")
      .when(col("VendorID") == 7, "Helix")
      .otherwise("Unknown")
      .alias("vendor"),
    
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    # Calculate trip duration in minutes
    (
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60
    ).alias("trip_duration"),
    "passenger_count",
    "trip_distance",

    when(col("RatecodeID") == 1, "Standard Rate")
      .when(col("RatecodeID") == 2, "JFK")
      .when(col("RatecodeID") == 3, "Newark")
      .when(col("RatecodeID") == 4, "Nassau or Westchester")
      .when(col("RatecodeID") == 5, "Negotiated Fare")
      .when(col("RatecodeID") == 6, "Group Ride")
      .otherwise("Unknown")
      .alias("rate_type"),
    
    "store_and_fwd_flag",
    col("PULocationID").alias("pu_location_id"),
    col("DOLocationID").alias("do_location_id"),
    
    when(col("payment_type") == 0, "Flex Fare trip")
      .when(col("payment_type") == 1, "Credit card")
      .when(col("payment_type") == 2, "Cash")
      .when(col("payment_type") == 3, "No charge")
      .when(col("payment_type") == 4, "Dispute")
      .when(col("payment_type") == 6, "Voided trip")
      .otherwise("Unknown")
      .alias("payment_type"),
    
    "fare_amount",
    "extra",
    "mta_tax",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
     col("Airport_fee").alias("airport_fee"),
    "cbd_congestion_fee",
    "load_timestamp"
)

print(f'Records Effected: {nyc_yellow_taxi_trans_df.count()}')

#### LOAD THE DATA INTO THE SILVER TABLE
- NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_CLEANSED

In [0]:
from datetime import datetime

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    TimestampType,
    DecimalType,
    DateType
)

# =====================================================
# WORKFLOW PARAMETERS
# =====================================================

dbutils.widgets.text("log_id", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("event_time", "")
dbutils.widgets.text("source_table", "")
dbutils.widgets.text("target_table", "")
dbutils.widgets.text("layer", "")
dbutils.widgets.text("notebook_path", "")
dbutils.widgets.text("pipeline_name", "")

# =====================================================
# RETRIEVE PARAMETERS
# =====================================================

log_id = dbutils.widgets.get("log_id")
run_id = dbutils.widgets.get("run_id")
raw_event_time = dbutils.widgets.get("event_time")

source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")
layer = dbutils.widgets.get("layer")

notebook_path = dbutils.widgets.get("notebook_path")
pipeline_name = dbutils.widgets.get("pipeline_name")

# =====================================================
# EVENT TIME
# =====================================================

try:
    if not raw_event_time or raw_event_time.startswith("{{"):
        event_time = datetime.now()
    else:
        event_time = datetime.fromisoformat(raw_event_time)
except Exception:
    event_time = datetime.now()

# =====================================================
# CURRENT USER
# =====================================================

user_name = spark.sql("SELECT current_user() AS user_name").first()["user_name"]

# =====================================================
# INITIALIZE AUDIT VARIABLES
# =====================================================

record_count = 0
status = "FAILED"
event_type = "LOAD_FAILURE"
message = ""

# =====================================================
# BUSINESS LOAD
# =====================================================

try:

    record_count = nyc_yellow_taxi_trans_df.count()

    # Target Write
    nyc_yellow_taxi_trans_df.write.mode('overwrite').saveAsTable('NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_CLEANSED')

    status = "SUCCESS"
    event_type = "LOAD_SUCCESS"
    message = f"Loaded {record_count} records into {target_table}"

except Exception as e:

    status = "FAILED"
    event_type = "LOAD_FAILURE"
    message = str(e)

# =====================================================
# CAPTURE END TIME
# =====================================================

load_end_time = datetime.now()

# =====================================================
# AUDIT SCHEMA
# =====================================================

audit_schema = StructType([

    StructField("log_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("event_time", DateType(), True),
    StructField("event_type", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("record_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("message", StringType(), True),
    StructField("user_name", StringType(), True),
    StructField("notebook_path", StringType(), True),
    StructField("pipeline_name", StringType(), True),
    StructField("load_start_time", TimestampType(), True),
    StructField("load_end_time", TimestampType(), True),
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("file_extension", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("source_folder", StringType(), True),
    StructField("file_size_bytes", LongType(), True),
    StructField("file_size_mb", DecimalType(18, 2), True),
    StructField("file_created_time", TimestampType(), True),
    StructField("file_modified_time", TimestampType(), True)

])

# =====================================================
# BUILD AUDIT RECORD
# =====================================================

audit_data = [(

    log_id,
    run_id,
    event_time,
    event_type,
    source_table,
    target_table,
    layer,
    record_count,
    status,
    message,
    user_name,
    notebook_path,
    pipeline_name,
    load_start_time,
    load_end_time,
    None,  # file_name
    None,  # file_path
    None,  # file_extension
    None,  # source_system
    None,  # source_folder
    None,  # file_size_bytes
    None,  # file_size_mb
    None,  # file_created_time
    None   # file_modified_time
)]

audit_df = spark.createDataFrame(
    audit_data,
    audit_schema
)

# =====================================================
# WRITE AUDIT LOG
# =====================================================

try:

    audit_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("NYCTAXI.AUDIT.PIPELINE_EXECUTION_LOG")

    print(f"Audit logging completed successfully for Run ID: {run_id}")

except Exception as audit_error:

    print(f"Business load completed but audit logging failed: {audit_error}")

# =====================================================
# FAIL NOTEBOOK IF LOAD FAILED
# =====================================================

if status == "FAILED":
    raise Exception(message)

In [0]:
dbutils.notebook.exit('YELLOW TAXI TRIP HAS BEEN LOADED INTO NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_CLEANSED')